In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from src.chess_features import (split_by_puzzle, topk, heuristic_score, fit_eval,
                                ORIGINAL_NUM, BASE_COLS, MATE_COLS, DYNAMIC_COLS,
                                NEW_COLS, CAT_COLS)

Path("results").mkdir(exist_ok=True)
cand = pd.read_csv("data/candidates.csv", dtype={"PuzzleId": str})
train, test = split_by_puzzle(cand)

print("Train puzzles:", train["PuzzleId"].nunique(), "| test puzzles:", test["PuzzleId"].nunique())
print("Puzzles in both:", len(set(train["PuzzleId"]) & set(test["PuzzleId"])))   # must be 0

Train puzzles: 4000 | test puzzles: 1000
Puzzles in both: 0


In [3]:
tr, val = split_by_puzzle(train, test_size=0.2, seed=0)
print("Train:", tr["PuzzleId"].nunique(), "| validation:", val["PuzzleId"].nunique())
print("Overlap:", len(set(tr["PuzzleId"]) & set(val["PuzzleId"])))   # must be 0

Train: 3200 | validation: 800
Overlap: 0


In [4]:
results = {}
results["Random"] = topk(test, np.zeros(len(test)))
results["Checks/captures heuristic"] = topk(test, heuristic_score(test).values)

res_logit, logit = fit_eval(train, test, ORIGINAL_NUM, CAT_COLS)
results["Logistic regression"] = res_logit

main_results = pd.DataFrame(results).T
main_results.to_csv("results/main_results.csv")

names = logit.named_steps["pre"].get_feature_names_out()
coefs = pd.Series(logit.named_steps["clf"].coef_[0], index=names).sort_values()
coefs.to_csv("results/coefficients.csv", header=["coefficient"])

main_results

,top1,top3
Random,0.051,0.125
Checks/captures heuristic,0.376,0.728
Logistic regression,0.543,0.818


In [5]:
dynamics = BASE_COLS + MATE_COLS + DYNAMIC_COLS

ablation = {
    "Heuristic features only": fit_eval(tr, val, BASE_COLS, [])[0],
    "+ mate":                  fit_eval(tr, val, BASE_COLS + MATE_COLS, [])[0],
    "+ opp_replies, attacked": fit_eval(tr, val, dynamics, [])[0],
    "+ piece type":            fit_eval(tr, val, dynamics, CAT_COLS)[0],
    "All original features":   fit_eval(tr, val, ORIGINAL_NUM, CAT_COLS)[0],
    "+ new tactical features": fit_eval(tr, val, ORIGINAL_NUM + NEW_COLS, CAT_COLS)[0],
}
ablation_df = pd.DataFrame(ablation).T
ablation_df.to_csv("results/ablation.csv")
ablation_df

,top1,top3
Heuristic features only,0.42750,0.75375
+ mate,0.49375,0.77625
"+ opp_replies, attacked",0.56500,0.82750
+ piece type,0.57500,0.83500
All original features,0.56375,0.82750
+ new tactical features,0.60625,0.85750


In [6]:
features = ORIGINAL_NUM + NEW_COLS

res_lr, _ = fit_eval(train, test, features, CAT_COLS, kind="logit")
res_mlp, mlp = fit_eval(train, test, features, CAT_COLS, kind="mlp")

nn_df = pd.DataFrame({"Logistic regression": res_lr, "Neural network (32,16)": res_mlp}).T
nn_df.to_csv("results/neural_network.csv")
nn_df

,top1,top3
Logistic regression,0.618,0.846
"Neural network (32,16)",0.662,0.873


In [7]:
rows = []
for seed in range(5):
    tr, te = split_by_puzzle(cand, seed=seed)
    rows.append({"seed": seed, "model": "Random", **topk(te, np.zeros(len(te)))})
    rows.append({"seed": seed, "model": "Heuristic", **topk(te, heuristic_score(te).values)})
    for name, kind in [("Logistic regression", "logit"), ("Neural network", "mlp")]:
        res, _ = fit_eval(tr, te, features, CAT_COLS, kind=kind)
        rows.append({"seed": seed, "model": name, **res})

repeated = pd.DataFrame(rows)
repeated.to_csv("results/repeated_splits.csv", index=False)
repeated.groupby("model")[["top1", "top3"]].agg(["mean", "std"]).round(3)

top1          top3       
                      mean    std   mean    std
model                                          
Heuristic            0.386  0.011  0.725  0.012
Logistic regression  0.608  0.009  0.839  0.015
Neural network       0.672  0.014  0.871  0.015
Random               0.054  0.012  0.152  0.010

In [8]:
wide = repeated.pivot(index="seed", columns="model", values=["top1", "top3"])
diff = pd.DataFrame({
    "top1": wide["top1"]["Neural network"] - wide["top1"]["Logistic regression"],
    "top3": wide["top3"]["Neural network"] - wide["top3"]["Logistic regression"],
})
print(diff.round(3))
print(diff.agg(["mean", "std"]).round(3))

       top1   top3
seed              
0     0.055  0.023
1     0.077  0.034
2     0.058  0.031
3     0.071  0.037
4     0.059  0.035
       top1   top3
mean  0.064  0.032
std   0.009  0.005


In [9]:
# Error analysis: Themes and Rating are used only AFTER prediction, never as features
features = ORIGINAL_NUM + NEW_COLS
_, nn = fit_eval(train, test, features, CAT_COLS, kind="mlp")

meta = pd.read_csv("data/puzzles_sample.csv", dtype=str, keep_default_na=False)
meta = meta[["PuzzleId", "Rating", "Themes"]].copy()
meta["Rating"] = pd.to_numeric(meta["Rating"])

t = test.copy()
t["_s"] = nn.predict_proba(t[features + CAT_COLS])[:, 1]
t = t.sample(frac=1, random_state=42)
t["rank"] = t.groupby("PuzzleId")["_s"].rank(ascending=False, method="first")

good = t[t["label"] == 1].merge(meta, on="PuzzleId")
good["top1"] = good["rank"] <= 1
good["top3"] = good["rank"] <= 3
print("Puzzles analysed:", len(good))

good["rating_bin"] = pd.qcut(good["Rating"], 4)
by_rating = good.groupby("rating_bin", observed=True)[["top1", "top3"]].mean()
by_rating.to_csv("results/errors_by_rating.csv")

ex = good.assign(theme=good["Themes"].str.split()).explode("theme")
by_theme = (ex.groupby("theme")
              .agg(n=("top1", "size"), top1=("top1", "mean"), top3=("top3", "mean")))
by_theme = by_theme[by_theme["n"] >= 30].sort_values("top1")
by_theme.to_csv("results/errors_by_theme.csv")

display(by_rating)
display(by_theme)

Puzzles analysed: 1000


,top1,top3
rating_bin,,
"(408.999, 1018.25]",0.920000,0.984000
"(1018.25, 1408.0]",0.792829,0.932271
"(1408.0, 1828.0]",0.496000,0.788000
"(1828.0, 3222.0]",0.437751,0.787149


,n,top1,top3
theme,,,
defensiveMove,53,0.415094,0.811321
pin,57,0.456140,0.736842
pawnEndgame,30,0.500000,0.766667
sacrifice,75,0.506667,0.880000
advancedPawn,56,0.517857,0.767857
veryLong,74,0.540541,0.851351
crushing,371,0.544474,0.811321
advantage,288,0.545139,0.819444
long,252,0.571429,0.845238
